# 🔥 Advanced · Nerfstudio (nerfacto)

> 🔥 **Advanced / heavy lab.** Your phone video → posed frames → a trained nerfacto NeRF.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **15–30 min** including downloads. Built on the official **[nerfstudio](https://docs.nerf.studio/)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Maps to lessons **B5–B7** — the production-grade counterpart to the from-scratch tiny-NeRF lab. Swap `nerfacto` for `splatfacto` to train Gaussians instead.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB | 1× RTX 3090/4090 / A100 ≥ 24 GB |
| **Storage** | video/images + outputs ~ 1–5 GB | ~10 GB / scene |
| **Time** | 15k iters ~ 10–20 min | 30k iters ~ 20–30 min (splatfacto similar) |

**Full pipeline (scale-up):** `ns-train nerfacto --data data/mine --max-num-iterations 30000` (or `splatfacto`).

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi

## 1 · Install

In [ ]:
!pip install -q nerfstudio
!pip install -q git+https://github.com/NVlabs/tiny-cuda-nn/#subdirectory=bindings/torch
!apt-get -qq install -y colmap ffmpeg

## 2 · Process your capture (extracts frames + estimates poses)
Upload a short, slow phone video as `my_capture.mp4`.

In [ ]:
!ns-process-data video --data my_capture.mp4 --output-dir data/mine --num-frames-target 150

## 3 · Train
(`--viewer.quit-on-train-completion True` so the cell returns; drop it to open the web viewer via a tunnel.)

In [ ]:
!ns-train nerfacto --data data/mine --max-num-iterations 15000 --viewer.quit-on-train-completion True

## 4 · Export a mesh / point cloud
Find the run dir under `outputs/` and export:

In [ ]:
import glob
cfg = sorted(glob.glob("outputs/**/config.yml", recursive=True))[-1]
print("config:", cfg)
!ns-export pointcloud --load-config "{cfg}" --output-dir exports/pcd --num-points 1000000 --remove-outliers True --normal-method open3d

## Evaluate — PSNR / SSIM / LPIPS on eval views

In [ ]:
!ns-eval --load-config outputs/<scene>/nerfacto/<run>/config.yml --output-path eval_results.json
import json; print(json.load(open("eval_results.json"))["results"])

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
These reconstructions become the **D · Scene / world** scene representation an agent reasons over.

## Next steps
- **Gaussian Splatting**: replace `nerfacto` with `splatfacto` (and `ns-export gaussian-splat`).
- Render a smooth camera path with `ns-render`.
- The web viewer streams over a Colab tunnel — see the Nerfstudio Colab docs.